# 06d · Phase 3: Cross-Architecture Analysis

Loads summary CSVs from shards 06a/06b/06c and produces the cross-architecture comparison.

**Prerequisites:** Mount Kaggle Datasets from 06a, 06b, 06c as inputs:
- `phase3-yolo26s-results`  → `/kaggle/input/phase3-yolo26s-results/`
- `phase3-rtdetr-results`   → `/kaggle/input/phase3-rtdetr-results/`
- `phase3-frcnn-results`    → `/kaggle/input/phase3-frcnn-results/`

On Colab, just point `EXPERIMENTS` at the Drive directory.

## Section 1 · Environment & Load Data

In [ ]:
# Environment detection
import os, sys
from pathlib import Path

ON_KAGGLE = os.path.exists('/kaggle/working')
ON_COLAB  = 'google.colab' in sys.modules or os.path.exists('/content')

if ON_KAGGLE:
    ROOT = Path('/kaggle/working')
    print('Runtime: Kaggle')
elif ON_COLAB:
    ROOT = Path('/content')
    print('Runtime: Colab')
else:
    ROOT = Path.cwd()
    print(f'Runtime: local ({ROOT})')

gpu = os.popen('nvidia-smi --query-gpu=name --format=csv,noheader').read().strip()
print(f'GPU: {gpu or "none detected"}')


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'ultralytics==8.4.9', 'pycocotools'], check=True)
print('Deps installed.')


In [ ]:
import pandas as pd, numpy as np, json as _json, os
from pathlib import Path

# Locate shard outputs
if ON_KAGGLE:
    SHARD_DIRS = {
        'yolo26s': Path('/kaggle/input/phase3-yolo26s-results/experiments_phase3'),
        'rtdetr':  Path('/kaggle/input/phase3-rtdetr-results/experiments_phase3'),
        'frcnn':   Path('/kaggle/input/phase3-frcnn-results/experiments_phase3'),
    }
else:  # Colab / local — all shards wrote to the same EXPERIMENTS dir
    ROOT = Path('/content') if os.path.exists('/content') else Path.cwd()
    _exp = ROOT / 'experiments_phase3'
    SHARD_DIRS = {'yolo26s': _exp, 'rtdetr': _exp, 'frcnn': _exp}

ANALYSIS_OUT = Path('/kaggle/working/analysis') if ON_KAGGLE else Path('/content/analysis')
ANALYSIS_OUT.mkdir(parents=True, exist_ok=True)

# Load per-arch summary CSVs
dfs = []
for arch, shard_dir in SHARD_DIRS.items():
    csv = shard_dir / f'summary_{arch}.csv'
    if csv.exists():
        df = pd.read_csv(csv)
        df['arch'] = arch
        dfs.append(df)
        print(f'Loaded {arch}: {len(df)} rows')
    else:
        print(f'[MISS] {csv}')

df_all = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
print(f'\nTotal runs loaded: {len(df_all)}')
df_all.head()


## Section 2 · Cross-Architecture Comparison Table

In [ ]:
# Pivot: rows=loss, cols=arch, values=mAP50-95 (clean split)
clean = df_all[df_all['split'] == 'clean'].copy()
pivot = clean.pivot_table(index='loss', columns='arch', values='map50_95')

# Delta vs EIoU per arch
if 'eiou' in pivot.index:
    delta = pivot.subtract(pivot.loc['eiou'], axis=1)
    delta.columns = [f'Δ_{c}' for c in delta.columns]
    table = pd.concat([pivot, delta], axis=1).round(4)
else:
    table = pivot.round(4)

print('=== Cross-Architecture mAP50-95 (clean split) ===')
print(table.to_string())
table.to_csv(ANALYSIS_OUT / 'cross_arch_map50_95.csv')


In [ ]:
# Success criterion: AEIoU beats EIoU on ≥ 2/3 architectures
aeiou_rows = pivot[pivot.index.str.startswith('aeiou')]
if not aeiou_rows.empty and 'eiou' in pivot.index:
    eiou_row = pivot.loc['eiou']
    best_aeiou = aeiou_rows.max()
    wins = (best_aeiou > eiou_row).sum()
    print(f'Best AEIoU vs EIoU per arch:')
    for arch in pivot.columns:
        diff = best_aeiou.get(arch, 0) - eiou_row.get(arch, 0)
        mark = '✓' if diff > 0 else '✗'
        print(f'  {arch:12s}: Δ={diff:+.4f}  {mark}')
    print(f'\n→ AEIoU wins on {wins}/3 architectures.')
    if wins >= 2:
        print('SUCCESS: gain is loss-driven, not architecture-specific.')
    else:
        print('INCONCLUSIVE: AEIoU gain not confirmed across architectures.')
else:
    print('AEIoU or EIoU rows missing — run training shards first.')


## Section 3 · Delta Bar Chart

In [ ]:
import matplotlib.pyplot as plt

if not aeiou_rows.empty and 'eiou' in pivot.index:
    best_lambda = aeiou_rows.mean(axis=1).idxmax()
    deltas = (pivot.loc[best_lambda] - pivot.loc['eiou']).dropna()

    fig, ax = plt.subplots(figsize=(7, 4))
    colors = ['#2ecc71' if v >= 0 else '#e74c3c' for v in deltas.values]
    ax.bar(deltas.index, deltas.values, color=colors, edgecolor='black')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_ylabel('ΔmAP50-95 vs EIoU (clean)')
    ax.set_title(f'Best AEIoU ({best_lambda}) − EIoU per Architecture')
    for i, (arch, v) in enumerate(deltas.items()):
        ax.text(i, v + (0.001 if v >= 0 else -0.002), f'{v:+.4f}',
                ha='center', va='bottom' if v >= 0 else 'top', fontsize=9)
    plt.tight_layout()
    plt.savefig(ANALYSIS_OUT / 'delta_bar_cross_arch.png', dpi=150)
    plt.show()
    print(f'Saved: {ANALYSIS_OUT}/delta_bar_cross_arch.png')


## Section 4 · mAP50-95 Heatmap (all losses × all archs)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

fig, ax = plt.subplots(figsize=(max(6, len(pivot.columns)*2), len(pivot)*0.6 + 1))
im = ax.imshow(pivot.values, cmap='RdYlGn', aspect='auto')
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, rotation=30, ha='right')
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        v = pivot.values[i, j]
        if not np.isnan(v):
            ax.text(j, i, f'{v:.3f}', ha='center', va='center', fontsize=8)
plt.colorbar(im, ax=ax, label='mAP50-95')
ax.set_title('Phase 3: mAP50-95 by Loss × Architecture (clean split)')
plt.tight_layout()
plt.savefig(ANALYSIS_OUT / 'heatmap_cross_arch.png', dpi=150)
plt.show()


## Section 5 · mAP50-95 vs Noise Level (per arch)

In [ ]:
# Line plot: how each loss degrades from clean → low → high noise, per arch
import matplotlib.pyplot as plt

SPLITS_ORDER = ['clean', 'low', 'high']
arch_list = df_all['arch'].unique()
fig, axes = plt.subplots(1, len(arch_list), figsize=(5*len(arch_list), 4), sharey=True)
if len(arch_list) == 1: axes = [axes]

for ax, arch in zip(axes, arch_list):
    sub = df_all[df_all['arch'] == arch]
    for loss_name in sub['loss'].unique():
        ldf = sub[sub['loss'] == loss_name]
        vals = [ldf[ldf['split']==s]['map50_95'].mean() for s in SPLITS_ORDER]
        ls = '--' if loss_name.startswith('aeiou') else '-'
        ax.plot(SPLITS_ORDER, vals, marker='o', linestyle=ls, label=loss_name)
    ax.set_title(arch)
    ax.set_ylabel('mAP50-95')
    ax.legend(fontsize=6)
plt.suptitle('mAP50-95 vs Noise Level by Architecture')
plt.tight_layout()
plt.savefig(ANALYSIS_OUT / 'noise_robustness_cross_arch.png', dpi=150)
plt.show()


## Section 6 · PR Curves (YOLOv26s + RT-DETR-L)

In [ ]:
# Load PR curve JSONs from Ultralytics shards
import matplotlib.pyplot as plt, json as _json

ARCH_LABELS = {'yolo26s': 'YOLOv26s', 'rtdetr': 'RT-DETR-L'}
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (arch, shard_dir) in zip(axes, [(a, SHARD_DIRS[a]) for a in ['yolo26s', 'rtdetr']
                                          if a in SHARD_DIRS]):
    metrics_dir = shard_dir / 'metrics'
    for loss_name, _ in [('eiou', None), ('ciou', None), ('eciou', None)]:
        run_name = f'phase3_{arch}_{loss_name}_clean_s42_e20'
        pr_path  = metrics_dir / run_name / 'pr_curve.json'
        if not pr_path.exists(): continue
        pr = _json.loads(pr_path.read_text())
        if pr['recall'] and pr['precision']:
            ax.plot(pr['recall'], pr['precision'], label=f'{loss_name} AP50={pr["ap50"]:.3f}')
    ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
    ax.set_title(f'PR Curve — {ARCH_LABELS.get(arch, arch)} (clean split)')
    ax.legend(fontsize=8); ax.set_xlim(0, 1); ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig(ANALYSIS_OUT / 'pr_curves_cross_arch.png', dpi=150)
plt.show()


## Section 7 · Final Save

In [ ]:
# Save full merged DataFrame
df_all.to_csv(ANALYSIS_OUT / 'phase3_all_results.csv', index=False)
n_figs = len(list(ANALYSIS_OUT.glob('*.png')))
n_csvs = len(list(ANALYSIS_OUT.glob('*.csv')))
print(f'Analysis complete.')
print(f'  Figures : {n_figs} PNGs')
print(f'  Tables  : {n_csvs} CSVs')
print(f'  Output  : {ANALYSIS_OUT}')
